<a href="https://colab.research.google.com/github/CharvRaj/RAJORACHARV/blob/main/RAG_UNDERSTANDING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip show transformers

Name: transformers
Version: 5.12.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: peft, sentence-transformers


In [ ]:
#Step 1 Installing all required libraries for the project

!pip install -q transformers sentence-transformers faiss-cpu pypdf python-docx nltk gradio pymupdf

# Importing basic Python modules
import os, io, math, gc, json, textwrap, re, random, collections

# Importing dataclass and typing utilities
from dataclasses import dataclass
from typing import List, Dict, Tuple

# Downloading NLTK resources for sentence tokenization
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# Importing sentence tokenizer
from nltk.tokenize import sent_tokenize

# Importing file upload utility for Google Colab
from google.colab import files

# Importing date and time module
from datetime import datetime

# Importing PDF reader
from pypdf import PdfReader

# Importing DOCX reader
from docx import Document as DocxDocument

# Importing embedding model
from sentence_transformers import SentenceTransformer

# Importing transformer models and tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Importing PyMuPDF for PDF processing
import fitz

# Importing FAISS for vector search
import faiss

# Importing NumPy for numerical operations
import numpy as np

# Importing Gradio for user interface
import gradio as gr

# Importing PyTorch
import torch

# Checking whether GPU is available
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Displaying current device information
DEVICE, torch.cuda.get_device_name(0) if DEVICE=="cuda" else "CPU"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 62.0 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


('cuda', 'Tesla T4')

In [ ]:
#Step 2:Upload all files which are required
from google.colab import files  #uploading files which are required in project  with extension like PDF,DOCX,TXT

uploaded = files.upload()

Saving Introduction_to_MachineLearning.docx to Introduction_to_MachineLearning.docx
Saving Introduction_to_ArtificialIntelligence.pdf to Introduction_to_ArtificialIntelligence.pdf
Saving Introduction_to_DeepLearning.txt to Introduction_to_DeepLearning.txt


In [ ]:
uploaded #to see uploaded files ,but content shown in bytes form ,in further step we will convert byte data to text data

{'Introduction_to_MachineLearning.docx': b'PK\x03\x04\x14\x00\x06\x00\x08\x00\x00\x00!\x002\x91oWf\x01\x00\x00\xa5\x05\x00\x00\x13\x00\x08\x02[Content_Types].xml \xa2\x04\x02(\xa0\x00\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\

In [ ]:
# Step 3: Validating uploaded files before processing

import os

# Maximum number of files allowed
MAX_FILES = 5

# Maximum file size allowed in MB
MAX_MB_PER_FILE = 15

# Supported file formats
ALLOWED_EXTS = {'.pdf', '.docx', '.txt'}

# Function to get file extension
def ext(name):
    return os.path.splitext(name)[1].lower()

# Checking if files are uploaded
assert len(uploaded) > 0, "No files uploaded."

# Checking upload limit
assert len(uploaded) <= MAX_FILES, f"Please upload at most {MAX_FILES} files."

# List to store valid files
docs_raw = []

# Validating each uploaded file
for fname, b in uploaded.items():

    # Checking file format
    assert ext(fname) in ALLOWED_EXTS, f"please upload {ALLOWED_EXTS} files"

    # Calculating file size
    Size_mb = len(b) / (1024 * 1024)

    # Checking file size limit
    assert Size_mb <= MAX_MB_PER_FILE, f"Please upload in size range {MAX_MB_PER_FILE}"

    # Storing valid files
    docs_raw.append((fname, b))

# Displaying validated files
docs_raw

[('Introduction_to_MachineLearning.docx',
  b'PK\x03\x04\x14\x00\x06\x00\x08\x00\x00\x00!\x002\x91oWf\x01\x00\x00\xa5\x05\x00\x00\x13\x00\x08\x02[Content_Types].xml \xa2\x04\x02(\xa0\x00\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

In [ ]:
# Step 4: Reading text content from TXT files

# Function to extract text from uploaded TXT file
def read_txt(bytes_blob: bytes) -> str:

    # Convert file bytes into readable text format
    return io.BytesIO(bytes_blob).read().decode('utf-8', errors='ignore')

In [ ]:
# Step 5: Extracting text from PDF files

# Function to read and extract text from a PDF file
def read_pdf(bytes_blob: bytes) -> str:

    # Creating PDF reader object
    reader = PdfReader(io.BytesIO(bytes_blob))

    # List to store text from all pages
    texts = []

    # Reading each page of the PDF
    for page in reader.pages:
        try:
            # Extracting text from the page
            texts.append(page.extract_text() or "")
        except:
            # Adding empty text if extraction fails
            texts.append("")

    # Combining all page texts into a single string
    return "\n".join(texts)

In [ ]:
# Step 6: Extracting text from DOCX files

# Function to read and extract text from a DOCX file
def read_docx(bytes_blob: bytes) -> str:

    # Converting file bytes into a readable format
    fh = io.BytesIO(bytes_blob)

    # Loading the DOCX document
    doc = DocxDocument(fh)

    # Combining text from all paragraphs
    return "\n".join([p.text for p in doc.paragraphs])

In [ ]:
# Step 7: Loading file content based on file extension

# Function to select the correct reader for each file type
def load_text_by_ext(fname: str, blob: bytes) -> str:

    # Getting the file extension
    ext = os.path.splitext(fname)[1].lower()

    # Reading TXT files
    if ext == '.txt':
        return read_txt(blob)

    # Reading PDF files
    elif ext == '.pdf':
        return read_pdf(blob)

    # Reading DOCX files
    elif ext == '.docx':
        return read_docx(blob)

    # Showing error for unsupported file types
    else:
        raise ValueError(f"Unsupported extension: {ext}")

In [ ]:
read_docx(docs_raw[0][1]) #calling function to see the content of file by using read_docx with passing docx_raw with [0]for file one [1] for extension

'Introduction to Machine Learning\nWhat is Machine Learning?\nMachine Learning (ML) is a branch of Artificial Intelligence that enables computers to learn from data and improve their performance without being explicitly programmed. Instead of following fixed instructions, machine learning systems identify patterns in data and use those patterns to make predictions or decisions.\nMachine Learning is widely used in applications such as recommendation systems, spam detection, image recognition, fraud detection, and medical diagnosis. As the amount of available data continues to grow, machine learning has become one of the most important technologies in modern computing.\n\nHow Machine Learning Works\nMachine learning begins with collecting data. The data is then cleaned and prepared for analysis. A machine learning algorithm is trained using this data so that it can learn patterns and relationships.\nAfter training, the model is tested using new data to evaluate its performance. If the mo

In [ ]:
load_text_by_ext(docs_raw[2][0], docs_raw[2][1])

'Introduction to Deep Learning\r\nWhat is Deep Learning?\r\nDeep Learning is a branch of Artificial Intelligence (AI) and Machine Learning (ML) that uses artificial neural networks with multiple layers to learn patterns from data. It is inspired by the structure and functioning of the human brain.\r\nTraditional machine learning algorithms often require manual feature extraction, whereas deep learning models can automatically learn important features directly from raw data. This ability makes deep learning highly effective for solving complex problems involving images, text, audio, and video.\r\nDeep learning has become one of the most important technologies in modern AI and powers many applications that people use every day.\r\n________________________________________\r\nRelationship Between AI, Machine Learning, and Deep Learning\r\nArtificial Intelligence is the broad field that focuses on creating intelligent systems.\r\nMachine Learning is a subset of Artificial Intelligence that 

In [ ]:
# Step 8: Cleaning and storing extracted text from documents

# List to store processed document text
docs_text = []

# Processing each uploaded document
for fname, blob in docs_raw:

    # Extracting text from the file
    text = load_text_by_ext(fname, blob)

    # Removing extra spaces before new lines
    text = re.sub(r'\s+\n', '\n ', text)

    # Removing multiple blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Removing unwanted spaces from start and end
    text = text.strip()

    # Storing document name and cleaned text
    docs_text.append({"name": fname, "text": text})

# Displaying document names and text length
for d in docs_text:
    print(d["name"], len(d["text"]))

Introduction_to_MachineLearning.docx 4947
Introduction_to_ArtificialIntelligence.pdf 5415
Introduction_to_DeepLearning.txt 6855


In [ ]:
# Step 9: Cleaning extracted text

# Function to remove unwanted spaces and special characters
def _clean_text(s: str) -> str:

    # Replacing multiple spaces with a single space
    s = re.sub(r'\s+', ' ', s).strip()

    # Removing special characters from the beginning and end
    s = re.sub(r'^\W+|\W+$', '', s).strip()

    # Returning cleaned text
    return s

In [ ]:
# Step 10: Extracting document title from PDF layout

# Function to identify the title from a PDF using font size and position
def extract_title_from_pdf_layout(pdf_bytes: bytes, consider_pages=3, top_band=0.35):

    # Opening the PDF file
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")

    # Checking if PDF contains pages
    if doc.page_count == 0:
        return None

    # Lists to store text spans and page heights
    spans, page_heights = [], {}

    # Reading the first few pages for title detection
    for pno in range(min(consider_pages, doc.page_count)):
        pg = doc.load_page(pno)
        page_heights[pno] = pg.rect.height

        # Extracting layout information
        data = pg.get_text("dict")

        # Reading text blocks, lines, and spans
        for b in data.get("blocks", []):
            for l in b.get("lines", []):
                for s in l.get("spans", []):

                    # Cleaning extracted text
                    txt = _clean_text(s.get("text", ""))

                    # Ignoring very short text
                    if not txt or len(txt) < 3:
                        continue

                    # Storing text properties
                    spans.append({
                        "page": pno,
                        "text": txt,
                        "size": float(s.get("size", 0)),
                        "bold": ("bold" in (s.get("font") or "").lower())
                                or ("black" in (s.get("font") or "").lower()),
                        "y0": s.get("bbox", [0, 0, 0, 0])[1]
                    })

    # Returning if no text is found
    if not spans:
        return None

    # Keeping text from the top portion of the page
    top_spans = [
        sp for sp in spans
        if page_heights.get(sp["page"], 1)
        and (sp["y0"] / page_heights[sp["page"]]) <= top_band
    ]

    if top_spans:
        spans = top_spans

    # Removing repeated headers
    per_text_pages = collections.defaultdict(set)

    for sp in spans:
        per_text_pages[sp["text"]].add(sp["page"])

    filtered = [
        sp for sp in spans
        if not (
            len(per_text_pages[sp["text"]]) >= 2
            and len(sp["text"]) <= 80
        )
    ]

    if filtered:
        spans = filtered

    # Grouping similar text lines together
    groups = []

    spans.sort(key=lambda x: (x["page"], -x["size"], x["y0"]))

    for sp in spans:
        placed = False

        for g in groups:
            if (
                g["page"] == sp["page"]
                and abs(g["size"] - sp["size"]) < 0.5
                and g["bold"] == sp["bold"]
            ):
                if abs(sp["y0"] - g["lines"][-1][1]) < 22:
                    g["lines"].append((sp["text"], sp["y0"]))
                    placed = True
                    break

        if not placed:
            groups.append({
                "page": sp["page"],
                "size": sp["size"],
                "bold": sp["bold"],
                "lines": [(sp["text"], sp["y0"])]
            })

    # Selecting the best title candidate
    for g in groups:
        lines_sorted = [
            t for t, _ in sorted(g["lines"], key=lambda x: x[1])
        ]

        cand = _clean_text(" ".join(lines_sorted))

        if len(cand) > 0:
            return cand

    # Using largest text as fallback title
    if spans:
        best = max(
            spans,
            key=lambda sp: (
                sp["size"],
                sp["bold"],
                -sp["y0"],
            ),
        )
        return best.get("text")

    return None

In [ ]:
title = extract_title_from_pdf_layout(docs_raw[1][1])
print(title)

Introduction to Artificial Intelligence


In [ ]:
# Step 11: Extracting title from DOCX document

# Function to find the most suitable title from a DOCX file
def extract_title_from_docx(docx_bytes: bytes):

    # Loading the DOCX document
    d = DocxDocument(io.BytesIO(docx_bytes))

    # Function to score each paragraph based on style and formatting
    def paragraph_score(p):

        # Cleaning paragraph text
        text = _clean_text(p.text)

        # Ignoring empty or very short text
        if not text or len(text) < 3:
            return None

        # Getting paragraph style name
        style_name = (p.style.name if p.style else "").lower()

        max_size_pt, any_bold = 0.0, False

        # Checking font size and bold formatting
        for run in p.runs:
            if run.font is not None:
                if run.font.size:
                    try:
                        max_size_pt = max(max_size_pt, float(run.font.size.pt))
                    except:
                        pass

                if run.font.bold:
                    any_bold = True

        # Assigning priority based on heading style
        style_priority = 0

        if "title" in style_name:
            style_priority = 3
        elif "heading 1" in style_name or style_name == "heading1":
            style_priority = 2
        elif "heading" in style_name:
            style_priority = 1

        return {
            "text": text,
            "style_priority": style_priority,
            "size": max_size_pt,
            "bold": any_bold
        }

    # Collecting title candidates
    candidates = []

    for p in d.paragraphs:
        sc = paragraph_score(p)
        if sc:
            candidates.append(sc)

    # Returning if no title candidate is found
    if not candidates:
        return None

    # Sorting candidates based on priority, size and formatting
    candidates.sort(
        key=lambda c: (
            c["style_priority"],
            c["size"],
            c["bold"],
            -len(c["text"])
        ),
        reverse=True
    )

    # Returning the best title candidate
    return candidates[0]["text"]

In [ ]:
print(extract_title_from_docx(docs_raw[0][1]))#by calling function to see the title of file that was extracted


Introduction to Machine Learning


In [ ]:
# Step 12: Extracting title from TXT file

# Function to get the title from the first non-empty line
def extract_title_from_txt_firstline(txt: str):

    # Checking each line in the text file
    for line in txt.splitlines():

        # Returning the first non-empty line as title
        if line.strip():
            return _clean_text(line)

    # Returning None if no valid title is found
    return None

In [ ]:
print(extract_title_from_txt_firstline(read_txt(docs_raw[2][1])))

Introduction to Deep Learning


In [ ]:
# Step 13: Detecting and storing document titles

# Dictionary to store document name and title
name_to_title = {}

# Creating a mapping of file names and file data
name_to_blob = {fname: blob for fname, blob in docs_raw}

# Processing each document
for item in docs_text:

    fname = item["name"]
    blob = name_to_blob[fname]

    # Getting file extension
    ext = os.path.splitext(fname)[1].lower()

    title = None

    try:
        # Extracting title from PDF file
        if ext == ".pdf":
            title = extract_title_from_pdf_layout(blob)

        # Extracting title from DOCX file
        elif ext == ".docx":
            title = extract_title_from_docx(blob)

        # Extracting title from TXT file
        elif ext == ".txt":
            title = extract_title_from_txt_firstline(item["text"])

    # Handling any extraction errors
    except Exception:
        title = None

    # Using file name as title if extraction fails
    if not title:
        title = os.path.splitext(fname)[0]

    # Storing detected title
    name_to_title[fname] = title

# Displaying detected titles
print("Detected titles:")

for d in docs_text:
    print(f"• {d['name']}  →  {name_to_title[d['name']]}")

Detected titles:
• Introduction_to_MachineLearning.docx  →  Introduction to Machine Learning
• Introduction_to_ArtificialIntelligence.pdf  →  Introduction to Artificial Intelligence
• Introduction_to_DeepLearning.txt  →  Introduction to Deep Learning


In [ ]:
# Step 14: Splitting document text into smaller chunks

# Function: chunk_text(text, target_chars, overlap_chars)
# text -> input document text
# target_chars -> maximum size of each chunk
# overlap_chars -> number of characters repeated between chunks

def chunk_text(text: str, target_chars: int = 1200, overlap_chars: int = 150) -> List[str]:

    # Splitting text into sentences
    sents = sent_tokenize(text)

    # List to store generated chunks
    chunks = []

    # Temporary buffer for building chunks
    buf = ""

    # Processing each sentence
    for s in sents:

        # Starting a new chunk
        if not buf:
            buf = s

        # Adding sentence if chunk size is within limit
        elif len(buf) + 1 + len(s) <= target_chars:
            buf += " " + s

        # Creating a new chunk when size limit is reached
        else:
            chunks.append(buf.strip())

            # Keeping some text overlap for better context
            tail = buf[-overlap_chars:]
            buf = (tail + " " + s).strip()

    # Adding the last chunk
    if buf:
        chunks.append(buf.strip())

    # Removing empty chunks and returning result
    return [c for c in chunks if len(c.strip()) > 0]

In [ ]:
# Step 15: Creating chunks for all documents

# List to store document names and their chunks
all_docs_chunks = []

# Processing each document
for d in docs_text:

    # Splitting document text into chunks
    ch = chunk_text(
        d["text"],
        target_chars=1400,   # Maximum characters in each chunk
        overlap_chars=200    # Overlapping characters between chunks
    )

    # Storing document name and generated chunks
    all_docs_chunks.append({
        "name": d["name"],
        "chunks": ch
    })

    # Displaying number of chunks created
    print(d["name"], "→", len(ch), "chunks")

Introduction_to_MachineLearning.docx → 5 chunks
Introduction_to_ArtificialIntelligence.pdf → 5 chunks
Introduction_to_DeepLearning.txt → 6 chunks


In [ ]:
# Step 16: Generating embeddings and building FAISS index

# Loading the sentence transformer model for text embeddings
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)

# Lists to store chunk text and metadata
corpus_texts, corpus_meta = [], []  # meta: (doc_id, doc_name, chunk_id)

# Flattening all document chunks into a single corpus
for doc_id, d in enumerate(all_docs_chunks):
    for chunk_id, c in enumerate(d["chunks"]):

        # Storing chunk text
        corpus_texts.append(c)

        # Storing metadata for each chunk
        corpus_meta.append((doc_id, d["name"], chunk_id))

# Batch size for embedding generation
BATCH = 64

# List to store embeddings
embs = []

# Generating embeddings in batches
for i in range(0, len(corpus_texts), BATCH):
    embs.extend(
        embedder.encode(
            corpus_texts[i:i+BATCH],
            normalize_embeddings=True,
            show_progress_bar=False
        )
    )

# Converting embeddings into NumPy array
embs = np.vstack(embs).astype('float32')

# Creating FAISS index for similarity search
# IndexFlatIP is used with normalized vectors for cosine similarity
index = faiss.IndexFlatIP(embs.shape[1])

# Adding embeddings to the index
index.add(embs)

# Displaying total vectors stored in the index
print("Index built with", index.ntotal, "vectors.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Index built with 16 vectors.


In [ ]:
# Step 17: Loading summarization and text generation models

# Checking whether GPU is available
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Setting device ID for model execution
device_id = 0 if DEVICE == "cuda" else -1

# -----------------------
# Summarization Model
# -----------------------

# Model used for generating document summaries
SUMM_MODEL = "facebook/bart-large-cnn"  # Alternative: sshleifer/distilbart-cnn-12-6

# Loading the summarization pipeline
summarizer = pipeline(
    task="text-generation",
    model=SUMM_MODEL,
    device=device_id
)

# -----------------------
# Question Answering / Text Generation Model
# -----------------------

# Model used for answering questions based on retrieved content
GEN_MODEL = "google/flan-t5-base"  # Alternative: google/flan-t5-small

# Loading the text generation pipeline
generator = pipeline(
    task="text-generation",
    model=GEN_MODEL,
    device=device_id
)

Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

[transformers] BartForCausalLM LOAD REPORT from: facebook/bart-large-cnn
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.self_attn.v_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.weight   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.q_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn_layer_norm.weight | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.bias                    | UNEXPECTED |  | 
mod

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

In [ ]:
# Step 18: Truncating long text safely

# Function: safe_truncate(txt, max_chars)
# txt -> input text
# max_chars -> maximum allowed characters

def safe_truncate(txt, max_chars=2400):

    # Returning original text if it is within the limit
    # Otherwise trimming the text and adding ellipsis
    return txt if len(txt) <= max_chars else txt[:max_chars] + "…"

In [ ]:
# Step 19: Generating summary for long documents

# Function: summarize_long_text(text, chunk_chars, overlap, map_max_len, map_min_len)
# text -> document text to summarize
# chunk_chars -> size of each text chunk
# overlap -> overlapping characters between chunks
# map_max_len -> maximum summary length for each chunk
# map_min_len -> minimum summary length for each chunk

def summarize_long_text(text: str, chunk_chars: int = 1200, overlap: int = 150,
                        map_max_len=256, map_min_len=60) -> str:

    # Splitting long document into smaller chunks
    parts = chunk_text(text, target_chars=chunk_chars, overlap_chars=overlap)

    # List to store summaries of individual chunks
    partial_summaries = []

    # Generating summary for each chunk using BART
    for p in parts if parts else [text]:

        # Limiting chunk size before summarization
        p_in = safe_truncate(p, 3500)

        # Generating summary
        out = summarizer(
            p_in,
            max_new_tokens=map_max_len,
            min_length=map_min_len,
            do_sample=False
        )[0]["generated_text"]

        partial_summaries.append(out.strip())

    # Combining all chunk summaries
    joined = safe_truncate(" ".join(partial_summaries), 4500)

    # Prompt for creating final concise bullet-point summary
    reduce_prompt = f"""Summarize the document below into 7–10 concise bullet points.
Each bullet must be specific and factual, and should include named entities (people, orgs, places), dates/years,
numbers/tables findings if present, and concrete outcomes or claims. Avoid generic sentences.

Document:
\"\"\"{joined}\"\"\"

Return only bullet points, each starting with '-'."""

    # Generating final summary using FLAN-T5
    final = generator(
        reduce_prompt,
        max_new_tokens=512,
        temperature=0.2
    )[0]["generated_text"]

    # Cleaning and formatting output as bullet points
    lines = [ln.strip() for ln in final.splitlines() if ln.strip()]

    bullets = []

    for ln in lines:
        if not ln.startswith("-"):
            ln = "- " + ln.lstrip("•*- ")

        bullets.append(ln)

    # Using fallback summary if too few bullets are generated
    if len(bullets) < 5:
        bullets = ["- " + s for s in partial_summaries[:8]]

    # Returning final summary with maximum 10 bullet points
    return "\n".join(bullets[:10]).strip()

In [ ]:
# Step 21: Generate and store a summary for each uploaded document
# Re-run to rebuild summaries with the new style
doc_summaries = {}
for d in docs_text:
    fname = d["name"]
    print(f"Summarizing: {name_to_title[fname]}")
    doc_summaries[fname] = summarize_long_text(d["text"])
print("Done.")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'min_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarizing: Introduction to Machine Learning


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer RobertaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max

Summarizing: Introduction to Artificial Intelligence


[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Summarizing: Introduction to Deep Learning


[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Done.


In [ ]:
# Step 20: Generating summaries for all uploaded documents

# Dictionary to store document summaries
doc_summaries = {}

# Processing each document
for d in docs_text:

    # Getting document name
    fname = d["name"]

    # Displaying current document being summarized
    print(f"Summarizing: {name_to_title[fname]}")

    # Generating and storing document summary
    doc_summaries[fname] = summarize_long_text(d["text"])

# Displaying completion message
print("Done.")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarizing: Introduction to Machine Learning


[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Summarizing: Introduction to Artificial Intelligence


[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Summarizing: Introduction to Deep Learning


[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Done.


In [ ]:
# Step 21: Displaying generated summaries for all documents

# Looping through each document summary
for fname, summary in doc_summaries.items():

    # Displaying document title
    print(f"--- Summary for {name_to_title[fname]} ---")

    # Displaying summary content
    print(summary)

    # Printing separator for better readability
    print("\n" + "=" * 80 + "\n")

--- Summary for Introduction to Machine Learning ---
- Summarize the document below into 7–10 concise bullet points.
- Each bullet must be specific and factual, and should include named entities (people, orgs, places), dates/years,
- numbers/tables findings if present, and concrete outcomes or claims. Avoid generic sentences.
- Document:
- """Introduction to Machine Learning
- What is Machine Learning? Machine Learning (ML) is a branch of Artificial Intelligence that enables computers to learn from data and improve their performance without being explicitly programmed. Instead of following fixed instructions, machine learning systems identify patterns in data and use those patterns to make predictions or decisions. Machine Learning is widely used in applications such as recommendation systems, spam detection, image recognition, fraud detection, and medical diagnosis. As the amount of available data continues to grow, machine learning has become one of the most important technologies in

In [ ]:
# Step 22: Retrieving the most relevant document chunks

# Function: search(query, top_k)
# query -> user question or search text
# top_k -> number of relevant chunks to retrieve

def search(query: str, top_k: int = 6) -> List[Tuple[float, str, Tuple[int, str, int]]]:

    # Converting the query into an embedding vector
    qe = embedder.encode([query], normalize_embeddings=True)

    # Searching the FAISS index for the most similar vectors
    # D -> similarity scores
    # I -> indices of matching chunks
    D, I = index.search(qe.astype('float32'), top_k)

    # List to store search results
    hits = []

    # Collecting retrieved chunks with their scores and metadata
    for score, idx in zip(D[0], I[0]):

        hits.append(
            (
                float(score),          # Similarity score
                corpus_texts[idx],     # Retrieved chunk text
                corpus_meta[idx]       # Document metadata
            )
        )

    # Returning top matching chunks
    return hits

In [ ]:
# Step 23: Restoring required components before FAQ generation

# This cell checks whether the answer prompt and search function are available

# Checking if answer prompt template exists
if "ANSWER_PROMPT_TMPL" not in globals():

    # Creating the default answer prompt template
    ANSWER_PROMPT_TMPL = """You are a helpful assistant. Answer the question strictly using the provided context.
If the answer is not in the context, say "I don't know from the provided documents."

Context:
{context}

Question: {question}

Answer:"""

# Checking if search function exists
if "search" not in globals():

    # Showing an error if search function is missing
    raise RuntimeError(
        "`search()` is missing. You MUST re-run the FAISS + embeddings cell that built the index and defined search()."
    )

# Displaying status message
print("ANSWER_PROMPT_TMPL restored. `search()` presence =", 'search' in globals())

 ANSWER_PROMPT_TMPL restored. `search()` presence = True


In [ ]:
print(globals().keys()) #to see the keys

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', '_', '__', '___', '_i', '_ii', '_iii', '_i1', '_exit_code', '_i2', 'os', 'io', 'math', 'gc', 'json', 'textwrap', 're', 'random', 'collections', 'dataclass', 'List', 'Dict', 'Tuple', 'nltk', 'sent_tokenize', 'files', 'datetime', 'PdfReader', 'DocxDocument', 'SentenceTransformer', 'AutoTokenizer', 'AutoModelForSeq2SeqLM', 'pipeline', 'fitz', 'faiss', 'np', 'gr', 'torch', 'DEVICE', '_2', '_i3', 'uploaded', '_i4', '_4', '_i5', 'MAX_FILES', 'MAX_MB_PER_FILE', 'ALLOWED_EXTS', 'ext', 'docs_raw', 'fname', 'b', 'Size_mb', '_5', '_i6', 'read_txt', '_i7', 'read_pdf', '_i8', 'read_docx', '_i9', 'load_text_by_ext', '_i10', '_10', '_i11', '_11', '_i12', 'docs_text', 'blob', 'text', 'd', '_i13', '_clean_text', '_i14', 'extract_title_from_pdf_layout', '_i15', 'title', '_i16', 'extract_title_from_docx', '_i17', '_i18', 'extract_title_

In [ ]:
# Step 24: Defining patterns to filter generic questions

# List of common generic question patterns
GENERIC_PATTERNS = [

    # Questions asking for main point
    r"\bmain point\b",

    # Questions asking why something is important
    r"\bwhy .* important\b",

    # Questions about target audience
    r"\bintended audience\b",

    # Questions about scope
    r"\bscope\b",

    # Questions about conclusion
    r"\bconclusion\b",

    # Questions asking for overview
    r"\boverview\b",

    # Questions asking for summary
    r"\bsum(mary|marize)\b",

    # Questions requesting additional details
    r"\badditional detail\??\b",

    # Generic document-related questions
    r"\bwhat is .*document\b",

    # Questions asking what the document does
    r"\bwhat does the document\b",
]

# Creating a regular expression pattern for matching generic questions
GENERIC_RE = re.compile("|".join(GENERIC_PATTERNS), re.I)

In [ ]:
# Step 25: Checking whether a generated question is specific enough

# Function: looks_specific(q)
# q -> generated question to be validated

def looks_specific(q: str) -> bool:

    # Removing extra spaces and question mark
    q = q.strip().rstrip("?")

    # Rejecting very short questions
    if len(q.split()) < 6:
        return False

    # Rejecting generic questions
    if GENERIC_RE.search(q):
        return False

    # Rejecting vague question beginnings
    if re.match(r"(?i)what is|what does|why is|who is$", q[:20]):
        return False

    # Returning True if question passes all checks
    return True

In [ ]:
# Step 26: Removing duplicate items while keeping original order

# Function: dedup_preserve_order(items)
# items -> list containing values that may have duplicates

def dedup_preserve_order(items):

    # Set to track already seen items
    seen = set()

    # List to store unique items
    out = []

    # Processing each item in the list
    for x in items:

        # Creating a normalized key for comparison
        key = re.sub(r"\s+", " ", x.strip().lower())

        # Skipping duplicate items
        if key in seen:
            continue

        # Adding unique item to the set and output list
        seen.add(key)
        out.append(x.strip())

    # Returning unique items in original order
    return out

In [ ]:

# Step 27: Generate high-quality, non-generic FAQ questions from a document summary
# --- Question generation from summary (sampling ON for variety) ---
FAQ_QS_PROMPT = """Write 12 distinct, specific FAQ questions that a reader of the following document would likely ask.
Questions must reference concrete people/organizations, locations, years/dates, metrics, methods, or outcomes from the text.
Avoid generic questions like "What is the main point?" or "Why is this important?".

Return ONLY the questions, one per line, no numbering or extra text.

Document:
\"\"\"{doc}\"\"\""""

def generate_varied_questions_from_summary(summary_text: str, k: int = 12) -> list:
    base = safe_truncate(summary_text, 4500)
    prompt = FAQ_QS_PROMPT.format(doc=base)
    out = generator(
        prompt,
        max_new_tokens=256,
        temperature=0.9,       # sampling for variety
        top_p=0.9,
        do_sample=True
    )[0]["generated_text"]
    qs = [q.strip().rstrip("?") + "?" for q in out.splitlines() if q.strip()]
    qs = dedup_preserve_order(qs)
    # filter generic
    qs = [q for q in qs if looks_specific(q)]
    # keep top ~10 and randomize a bit to avoid same 5 every run
    random.shuffle(qs)
    return qs[:max(k, 5)]

In [ ]:
# Step 28: Answer a question using only one selected document (robust retrieval + fallback)
# --- Answering with robust context (doc retrieval + summary fallback) ---
def answer_within_doc_robust(question: str, doc_name: str, summary_text: str) -> str:
    # primary: restrict hits to this doc
    hits = search(question, top_k=12)
    hits = [h for h in hits if h[2][1] == doc_name]
    ctx_blocks, seen = [], set()
    for sc, txt, meta in sorted(hits, key=lambda x: -x[0]):
        if len(" ".join(ctx_blocks)) > 4200: break
        sig = hash(txt)
        if sig in seen:
            continue
        seen.add(sig); ctx_blocks.append(txt)

    # fallback: if retrieval is weak, include summary
    context = "\n\n".join(ctx_blocks)
    if len(context) < 400 and summary_text:
        # prepend summary to ensure we have something factual
        context = (summary_text.strip() + "\n\n" + context).strip()

    if not context:
        # absolute fallback to summary only (better than “Not specified”)
        context = summary_text if summary_text else "No context."

    prompt = ANSWER_PROMPT_TMPL.format(context=context, question=question)
    ans = generator(
        prompt,
        max_new_tokens=220,
        temperature=0.2,   # deterministic-ish answers
        do_sample=False
    )[0]["generated_text"].strip()

    # last resort: if model still returns something empty or evasive, try a 2nd pass using only summary
    if (not ans) or len(ans.split()) < 3 or "I don't know" in ans:
        if summary_text:
            prompt2 = ANSWER_PROMPT_TMPL.format(context=summary_text, question=question)
            ans2 = generator(prompt2, max_new_tokens=200, temperature=0.2, do_sample=False)[0]["generated_text"].strip()
            if ans2 and "I don't know" not in ans2:
                ans = ans2

    # keep it tight
    return ans

In [ ]:

# Step 29: Create an FAQ generator for one document (make 5 good questions and answer them)
# --- Build FAQs using title for display, filename for retrieval mapping ---
def generate_faqs_for_document_v2(doc_name: str, title: str, text: str, summary: str) -> list:
    # ensure we have a seed summary; if not, synthesize a short one
    seed = summary if (summary and len(summary) > 60) else summarize_long_text(text)

    # 1) make 12 candidates from summary
    candidates = generate_varied_questions_from_summary(seed, k=12)

    # 2) if we somehow have <5, synthesize from top bullets
    if len(candidates) < 5:
        bullets = [ln[2:].strip() for ln in seed.splitlines() if ln.strip().startswith("- ")]
        for b in bullets:
            if len(candidates) >= 5: break
            frag = " ".join(b.split()[:12])
            candidates.append(f"What specific findings does the document report about {frag}?")
        candidates = dedup_preserve_order(candidates)

    # 3) pick 5 best-looking questions (stable order but de-generic)
    final_qs = candidates[:5]

    # 4) answer each question robustly
    faqs = []
    for q in final_qs:
        a = answer_within_doc_robust(q, doc_name, summary)
        faqs.append((q, a))
    return faqs

In [ ]:

# Step 30: Generate FAQs for every uploaded document and store them in a dictionary
# --- Run for all docs (uses previously computed: docs_text, name_to_title, doc_summaries) ---
doc_faqs = {}
for d in docs_text:
    fname = d["name"]
    print("Generating FAQs for", name_to_title[fname])
    doc_faqs[fname] = generate_faqs_for_document_v2(fname, name_to_title[fname], d["text"], doc_summaries.get(fname))
print("Done (Improved FAQ Generation v2).")


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'top_p', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating FAQs for Introduction to Machine Learning


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_to

Generating FAQs for Introduction to Artificial Intelligence


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Generating FAQs for Introduction to Deep Learning


[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Done (Improved FAQ Generation v2).


In [ ]:
# Step 31: Create the main Q&A function (retrieve chunks, build context, generate answer, show sources)
ANSWER_PROMPT_TMPL = """You are a helpful assistant. Answer the question strictly using the provided context.
If the answer is not in the context, say "I don't know from the provided documents."

Context:
{context}

Question: {question}

Answer:"""

def answer_question(question: str, top_k: int = 6, max_ctx_chars: int = 4200) -> Dict:
    hits = search(question, top_k=top_k)
    # merge contexts, sorted by score descending
    ctx_blocks = []
    seen = set()
    for sc, txt, meta in sorted(hits, key=lambda x: -x[0]):
        if len(" ".join(ctx_blocks)) > max_ctx_chars: break
        # avoid exact dupes
        sig = hash(txt)
        if sig in seen:
            continue
        seen.add(sig)
        ctx_blocks.append(txt)
    context = "\n\n".join(ctx_blocks)
    prompt = ANSWER_PROMPT_TMPL.format(context=context, question=question)
    out = generator(prompt, max_new_tokens=256, temperature=0.2)[0]["generated_text"].strip()
    # include source pointers
    sources = [{"doc_name": m[1], "chunk_id": m[2], "score": float(s)} for (s,_,m) in hits[:min(top_k,5)]]
    return {"answer": out, "sources": sources, "retrieved_snippets": [t for _,t,_ in hits[:3]]}


In [ ]:
# Step 32: Print a complete overview for each document (name, summary, and FAQs)
def show_document_overview():
    for d in docs_text:
        name = d["name"]
        print("="*100)
        print(name)
        print("- Summary -")
        print(textwrap.fill(doc_summaries[name], width=100))
        print("\n- FAQs -")
        for i,(q,a) in enumerate(doc_faqs[name],1):
            print(f"Q{i}: {q}\nA{i}: {a}\n")

show_document_overview()


Introduction_to_MachineLearning.docx
- Summary -
- Summarize the document below into 7–10 concise bullet points. - Each bullet must be specific and
factual, and should include named entities (people, orgs, places), dates/years, - numbers/tables
findings if present, and concrete outcomes or claims. Avoid generic sentences. - Document: -
"""Introduction to Machine Learning - What is Machine Learning? Machine Learning (ML) is a branch of
Artificial Intelligence that enables computers to learn from data and improve their performance
without being explicitly programmed. Instead of following fixed instructions, machine learning
systems identify patterns in data and use those patterns to make predictions or decisions. Machine
Learning is widely used in applications such as recommendation systems, spam detection, image
recognition, fraud detection, and medical diagnosis. As the amount of available data continues to
grow, machine learning has become one of the most important technologies in mod

In [ ]:

# Step 33: Create the Q&A controller for the Gradio UI
# ---- D) Gradio UI with Titles ----

TITLE_TO_NAME = {name_to_title[d["name"]]: d["name"] for d in docs_text}

def qa_interface(question, title_scope):
    # map chosen title to original filename
    if title_scope != "All": #This checks if the user selected a specific document instead of searching all documents.
        doc_scope = TITLE_TO_NAME[title_scope]
        # Restrict to that doc
        hits = search(question, top_k=12)
        hits = [h for h in hits if h[2][1] == doc_scope]
        ctx = []
        seen = set()
        for sc, txt, meta in sorted(hits, key=lambda x: -x[0]):
            if len(" ".join(ctx)) > 4200: break
            sig = hash(txt)
            if sig in seen: continue
            seen.add(sig); ctx.append(txt)
        context = "\n\n".join(ctx) if ctx else "No context available."
        prompt = ANSWER_PROMPT_TMPL.format(context=context, question=question)
        out = generator(prompt, max_new_tokens=256, temperature=0.2)[0]["generated_text"].strip()
        sources = [{"title": name_to_title[m[1]], "chunk_id": m[2], "score": float(s)} for (s,_,m) in hits[:5]]
        return out, json.dumps(sources, indent=2)
    else:
        res = answer_question(question, top_k=8)
        # decorate sources with titles
        for s in res["sources"]:
            s["title"] = name_to_title.get(s["doc_name"], s["doc_name"])
            del s["doc_name"]
        return res["answer"], json.dumps(res["sources"], indent=2)

In [ ]:
# Step 34: Show the correct summary based on the selected scope (one document or all)
def get_summary(title_scope):
    if title_scope == "All":
        merged = []
        for d in docs_text:

            fname = d["name"]
            merged.append(f"### {name_to_title[fname]}\n{doc_summaries[fname]}")
        return "\n\n".join(merged)
    fname = TITLE_TO_NAME[title_scope]
    return doc_summaries.get(fname, "No summary available.")

In [ ]:
#Step 35: Display FAQs based on the selected scope (one document or all documents)

# Function: get_faqs(title_scope)
# title_scope -> selected document title or "All"

def get_faqs(title_scope):

    # Checking if all document FAQs are requested
    if title_scope == "All":

        # List to store FAQs of all documents
        out = []

        # Looping through all documents
        for d in docs_text:
            fname = d["name"]
            friendly = name_to_title[fname]

            # Adding document title and FAQs
            out.append("### " + friendly + "\n" + "\n".join([f"Q{i+1}: {q}\nA{i+1}: {a}" for i,(q,a) in enumerate(doc_faqs[fname])]))

        # Returning FAQs of all documents
        return "\n\n".join(out)

    # Getting file name of selected document
    fname = TITLE_TO_NAME[title_scope]

    # Getting FAQs of selected document
    faqs = doc_faqs.get(fname, [])

    # Returning FAQs or "No FAQs." if none are available
    return "\n".join([f"Q{i+1}: {q}\nA{i+1}: {a}" for i,(q,a) in enumerate(faqs)]) or "No FAQs."

In [ ]:
# Step 36: Building the Gradio web application

# List of document titles for the dropdown menu
DOC_TITLES = ["All"] + list(TITLE_TO_NAME.keys())

# Creating the Gradio interface
with gr.Blocks(title="Mini NotebookLM (Local RAG)") as demo:

    # Displaying the application title
    gr.Markdown("# 📚 Mini NotebookLM (Local RAG) – Colab\nUpload ➜ Summaries & FAQs ➜ Ask Questions, grounded in your docs.")

    # Creating a row for document selection
    with gr.Row():

        # Dropdown to select a document or all documents
        title_scope = gr.Dropdown(
            DOC_TITLES,
            value="All",
            label="Document Scope (by Title)"
        )

    # -------------------- Q&A Tab --------------------
    with gr.Tab("Ask Questions"):

        # Textbox for entering a question
        question = gr.Textbox(
            label="Your question",
            placeholder="Ask something grounded in your uploaded docs…"
        )

        # Button to generate answer
        btn = gr.Button("Answer")

        # Textbox to display answer
        answer = gr.Textbox(label="Answer", lines=8)

        # Textbox to display retrieved source chunks
        sources = gr.Textbox(
            label="Top Source Chunks (title, chunk_id, score)",
            lines=8
        )

        # Connecting button with question-answer function
        btn.click(
            qa_interface,
            inputs=[question, title_scope],
            outputs=[answer, sources]
        )

    # -------------------- Summary Tab --------------------
    with gr.Tab("Summaries"):

        # Button to display document summary
        btn_sum = gr.Button("Show Summary")

        # Textbox to display summary
        sum_box = gr.Textbox(label="Summary", lines=20)

        # Connecting summary button with function
        btn_sum.click(
            get_summary,
            inputs=[title_scope],
            outputs=[sum_box]
        )

    # -------------------- FAQ Tab --------------------
    with gr.Tab("FAQs"):

        # Button to display FAQs
        btn_faq = gr.Button("Show FAQs (5)")

        # Textbox to display FAQs
        faq_box = gr.Textbox(label="FAQs", lines=20)

        # Connecting FAQ button with function
        btn_faq.click(
            get_faqs,
            inputs=[title_scope],
            outputs=[faq_box]
        )

# Launching the web application
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7a3d577698c2ed9d43.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
